# Raster / PSTH by value latent for opto-tagged units

Generates per-unit **rasters + quantile PSTH** (same style as `plot_raster copy` /
`raster_worker.process_session`) binned by the **`QLearning_L1F1_CK1_softmax`** value
latents, but restricted to the **opto-tagged units** selected in
`save_tagged_unit_figures.ipynb`.

Pipeline:
1. **Tagged units** — read the saved opto-tagging metrics CSV and re-run
   `select_tagged_units` (standalone) to get the tagged unit ids for the target condition.
2. **Session resources** — load the Zarr PSTH, behavior-summary CSV and trial ids
   (same file conventions as `raster_worker`).
3. **Plot** — for each value latent, call `plot_raster_and_quantile_psth_by_latent`
   with `unit_ids = tagged_units`.

The opto-tagging `unit_id` (index into the NWB units table) matches the Zarr
`unit_index`, so tagged ids can be passed straight through as `unit_ids`.

In [9]:
# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Analysis modules loaded from: /root/capsule/src/aind_dft_ephys_analysis


In [10]:
# =============================================================================
# 2. CONFIG
# =============================================================================
from pathlib import Path

# --- Session identifiers -----------------------------------------------------
# session_date : short NWB/opto-tagging session id (matches the metrics CSV name).
# ephys_session: full sorted folder name used by the PSTH/behavior pipeline.
#                Leave as None to auto-discover it from the PSTH Zarr in step 4.
session_date = '863973_2026-08-25_14-18-29'
ephys_session = None

# --- Paths (match raster_worker.py conventions) ------------------------------
PSTH_DIR = Path("/root/capsule/scratch/psth_results")
RESULTS_DIR = Path("/root/capsule/scratch/behavior_summary")
metrics_csv_path = f"/root/capsule/scratch/opto_tagging/metrics_{session_date}.csv"
OUTDIR = Path(f"/root/capsule/scratch/raster_plot_tagged/{session_date}")

# --- Value latents for the QLearning_L1F1_CK1_softmax model ------------------
# (column name in the behavior summary, colorbar label). Missing columns are
# skipped automatically in step 5.
MODEL = "QLearning_L1F1_CK1_softmax"
VALUE_LATENTS = [
    (f"{MODEL}-sumQ-1",    "sumQ (total value)"),
    (f"{MODEL}-deltaQ-1",  "deltaQ (value difference)"),
    (f"{MODEL}-chosenQ-1", "chosenQ (chosen value)"),
]

# --- Raster / PSTH alignment -------------------------------------------------
ALIGN_TO = "go_cue"
TIME_WIN = (-3, 4)

# --- Opto-tag condition + selection (must match save_tagged_unit_figures) ----
#   (target_power, location_tag, laser_name, cycle_duration, frequency, pulse_duration)
target_cond = (5.0, 1.0, "Laser_1", 1.0, 10.0, 0.05)
SELECT_PULSE_INDEX = -1   # -1 = pooled over all pulses in the train

print(f"Metrics CSV : {metrics_csv_path}")
print(f"Output dir  : {OUTDIR}")
print(f"Value latents: {[c for c, _ in VALUE_LATENTS]}")

Metrics CSV : /root/capsule/scratch/opto_tagging/metrics_863973_2026-08-25_14-18-29.csv
Output dir  : /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29
Value latents: ['QLearning_L1F1_CK1_softmax-sumQ-1', 'QLearning_L1F1_CK1_softmax-deltaQ-1', 'QLearning_L1F1_CK1_softmax-chosenQ-1']


In [11]:
# =============================================================================
# 3. TAGGED UNITS (re-select from the saved opto-tagging metrics CSV)
# =============================================================================
# compute_tagging_metrics only ran on QC-passing units, so every unit_id in the
# metrics table already passed QC -> selecting tagged==True is sufficient.
import re
import pandas as pd
from ast import literal_eval
from optical_tagging import select_tagged_units

metrics_df = pd.read_csv(metrics_csv_path)

tagged_df = select_tagged_units(
    metrics_df,
    alpha=0.05,
    effect_ratio=2.0,
    min_abs_increase=0.0,
    min_reliability=0.2,
    max_latency=0.006,
    max_jitter=0.003,
    correction="fdr_bh",
)

# `condition` is a string after the CSV round-trip. Older CSVs may contain numpy
# scalar wrappers like "np.float64(1.0)"; strip them before parsing.
def as_tuple(c):
    if isinstance(c, str):
        c = re.sub(r"np\.\w+\(([^()]*)\)", r"\1", c)
        try:
            c = literal_eval(c)
        except (ValueError, SyntaxError):
            return None
    return tuple(c) if isinstance(c, (tuple, list)) else None

# Match the condition on every field EXCEPT laser_name (index 2).
IGNORE_IDX = 2
match_idx = [i for i in range(6) if i != IGNORE_IDX]

def matches_ignore_laser(c):
    c = as_tuple(c)
    if c is None or len(c) != 6:
        return False
    return all(c[i] == target_cond[i] for i in match_idx)

cond_mask = tagged_df["condition"].apply(matches_ignore_laser)
pulse_mask = tagged_df["pulse_index"] == SELECT_PULSE_INDEX
sigrows = tagged_df[cond_mask & pulse_mask & (tagged_df["tagged"] == True)]

tagged_units = sorted(int(u) for u in sigrows["unit_id"].unique())
print(f"Condition (laser_name ignored): {target_cond} | pulse_index={SELECT_PULSE_INDEX}")
print(f"Tagged units ({len(tagged_units)}): {tagged_units}")
if not tagged_units:
    raise ValueError(
        "No tagged units found for this condition/selection. "
        "Check that target_cond matches a condition present in the metrics CSV."
    )

Condition (laser_name ignored): (5.0, 1.0, 'Laser_1', 1.0, 10.0, 0.05) | pulse_index=-1
Tagged units (122): [310, 311, 748, 752, 759, 788, 792, 2023, 2025, 2036, 2059, 2109, 2115, 2116, 2149, 2185, 2200, 2203, 2208, 2209, 2212, 2215, 2225, 2230, 2233, 2238, 2239, 2240, 2241, 2245, 2250, 2255, 2257, 2259, 2263, 2264, 2272, 2277, 2284, 2288, 2289, 2296, 2300, 2303, 2314, 2315, 2317, 2322, 2324, 2326, 2331, 2334, 2344, 2366, 2558, 2586, 2587, 2588, 2609, 2613, 2629, 2630, 2634, 2685, 2689, 2692, 2706, 2732, 2735, 2803, 2811, 2826, 2845, 2946, 2947, 2960, 2983, 3000, 3014, 3020, 3023, 3031, 3032, 3033, 3258, 3269, 3326, 3329, 3335, 3336, 3385, 3393, 3400, 3403, 3453, 3467, 3502, 3603, 3620, 3629, 3656, 3663, 3665, 3673, 3696, 3700, 3705, 3718, 3736, 3757, 3809, 3814, 4045, 4062, 4077, 4106, 4159, 4227, 4270, 4273, 4280, 4320]


In [12]:
# =============================================================================
# 4. LOAD SESSION RESOURCES (Zarr PSTH, behavior summary, trials)
# =============================================================================
import numpy as np
from general_utils import smart_read_csv
from behavior_utils import find_trials, get_fitted_model_names
from nwb_utils import NWBUtils
from create_psth import load_zarr

# Auto-discover the full sorted session name from the PSTH Zarr if not set.
if ephys_session is None:
    cands = sorted(PSTH_DIR.glob(f"ecephys_{session_date}_sorted*_0.2s.zarr"))
    if not cands:
        raise FileNotFoundError(
            f"No PSTH Zarr matching ecephys_{session_date}_sorted*_0.2s.zarr in {PSTH_DIR}"
        )
    ephys_session = cands[0].name[: -len("_0.2s.zarr")]
print(f"ephys_session: {ephys_session}")

zarr_path = PSTH_DIR / f"{ephys_session}_0.2s.zarr"
beh_csv = RESULTS_DIR / f"behavior_summary-{ephys_session}.csv"
assert zarr_path.exists(), f"Zarr not found: {zarr_path}"
assert beh_csv.exists(), f"Behavior CSV not found: {beh_csv}"

print(f"Fitted models: {get_fitted_model_names(session_name=ephys_session)}")

nwb, _ = NWBUtils.combine_nwb(session_name=ephys_session)
psth = load_zarr(str(zarr_path))
beh = smart_read_csv(str(beh_csv))
trials = np.asarray(find_trials(nwb, "response"))
print(f"n_trials (response): {len(trials)}")

ephys_session: ecephys_863973_2026-08-25_14-18-29_sorted_2026-09-09_23-04-54
[Attempt 1/3] Network failure for session ecephys_863973_2026-08-25_14-18-29_sorted_2026-09-09_23-04-54: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Fitted models: ['ForagingCompareThreshold', 'WSLS', 'QLearning_L2F1_CKfull_softmax', 'QLearning_L1F1_CK1_softmax', 'QLearning_L1F0_epsi', 'QLearning_L2F1_softmax']
Found ephys NWB: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29_sorted_2026-09-09_23-04-54/nwb/ecephys_863973_2026-08-25_14-18-29_experiment1_recording1.nwb
Successfully read ephys NWB from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29_sorted_2026-09-09_23-04-54/nwb/ecephys_863973_2026-08-25_14-18-29_experiment1_recording1.nwb
Found behavior NWB: /root/capsule/data/behavior_nwb/863973_2026-08-25_14-18-29.nwb
Successfully read behavior NWB from: /root/capsule/data/behavior_nwb/863973_2026-08-25_14-18-29.nwb
Successfully appended units table

In [13]:
# =============================================================================
# 5. PLOT RASTER + QUANTILE PSTH BY VALUE LATENT (tagged units only)
# =============================================================================
import matplotlib.pyplot as plt
from plot_raster import plot_raster_and_quantile_psth_by_latent

def _as_1d_float_array(x):
    arr = np.asarray(x, dtype=np.float32)
    return arr if arr.ndim == 1 else arr.reshape(-1)

OUTDIR.mkdir(parents=True, exist_ok=True)
n_trials = int(len(trials))

for col, lname in VALUE_LATENTS:
    if col not in beh.columns:
        print(f"skip: {col} (missing column)")
        continue
    vals = _as_1d_float_array(beh.at[0, col])
    if vals.shape[0] != n_trials:
        print(f"skip: {col} (length {vals.shape[0]} != trials {n_trials})")
        continue

    try:
        plot_raster_and_quantile_psth_by_latent(
            source=psth,
            latent_values=vals,
            latent_trial_ids=trials,
            unit_ids=tagged_units,          # restrict to opto-tagged units
            align_to_event=ALIGN_TO,
            time_window=TIME_WIN,
            n_bins=4,
            binning="equal",
            quantile_stat="mean",
            ci="sem",
            title_prefix="",
            figsize=(6, 5),
            save_path=str(OUTDIR),
            cmap_name="coolwarm",
            raster_colormap=False,          # continuous latent -> black rasters
            show_colormap=True,
            save_prefix=f"{col}_",
            latent_name=lname,
            show=False,
            overwrite=True,
            min_trial_rate=1,
            save_format=["png"],
        )
        print(f"plotted: {col} for {len(tagged_units)} tagged units -> {OUTDIR}")
    except Exception as e:
        print(f"error plotting {col}: {e}")
    finally:
        plt.close("all")

print("Done.")

Saved Unit 310 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-sumQ-1/QLearning_L1F1_CK1_softmax-sumQ-1_unit_310.png
Saved Unit 311 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-sumQ-1/QLearning_L1F1_CK1_softmax-sumQ-1_unit_311.png
Saved Unit 748 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-sumQ-1/QLearning_L1F1_CK1_softmax-sumQ-1_unit_748.png
Saved Unit 752 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-sumQ-1/QLearning_L1F1_CK1_softmax-sumQ-1_unit_752.png
Saved Unit 759 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-sumQ-1/QLearning_L1F1_CK1_softmax-sumQ-1_unit_759.png
Saved Unit 788 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-sumQ-1/QLe

/opt/conda/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/root/capsule/src/aind_dft_ephys_analysis/plot_raster.py:834: RuntimeWarning: All-NaN slice encountered
  fmax = np.nanmax(upper if upper is not None else center)


Saved Unit 3023 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-sumQ-1/QLearning_L1F1_CK1_softmax-sumQ-1_unit_3023.png
Saved Unit 3031 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-sumQ-1/QLearning_L1F1_CK1_softmax-sumQ-1_unit_3031.png
Saved Unit 3032 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-sumQ-1/QLearning_L1F1_CK1_softmax-sumQ-1_unit_3032.png
Saved Unit 3033 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-sumQ-1/QLearning_L1F1_CK1_softmax-sumQ-1_unit_3033.png
Saved Unit 3258 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-sumQ-1/QLearning_L1F1_CK1_softmax-sumQ-1_unit_3258.png
Saved Unit 3269 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax

/opt/conda/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/root/capsule/src/aind_dft_ephys_analysis/plot_raster.py:834: RuntimeWarning: All-NaN slice encountered
  fmax = np.nanmax(upper if upper is not None else center)


Saved Unit 3023 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-deltaQ-1/QLearning_L1F1_CK1_softmax-deltaQ-1_unit_3023.png
Saved Unit 3031 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-deltaQ-1/QLearning_L1F1_CK1_softmax-deltaQ-1_unit_3031.png
Saved Unit 3032 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-deltaQ-1/QLearning_L1F1_CK1_softmax-deltaQ-1_unit_3032.png
Saved Unit 3033 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-deltaQ-1/QLearning_L1F1_CK1_softmax-deltaQ-1_unit_3033.png
Saved Unit 3258 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-deltaQ-1/QLearning_L1F1_CK1_softmax-deltaQ-1_unit_3258.png
Saved Unit 3269 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearn

Saved Unit 4062 figure to /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29/QLearning_L1F1_CK1_softmax-deltaQ-1/QLearning_L1F1_CK1_softmax-deltaQ-1_unit_4062.png
